# TAREAS

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import time

def Tarea():
    # Lista para almacenar todos los enlaces filtrados
    todos_los_enlaces = []

    # Inicializar el navegador una vez
    driver = webdriver.Chrome()

    # Loop de páginas del 1 al 6
    for pagina in range(1, 7):
        url = f'https://www.bumeran.com.pe/en-lima/empleos-area-tecnologia-sistemas-y-telecomunicaciones-subarea-programacion-full-time-publicacion-menor-a-15-dias.html?page={pagina}'
        driver.get(url)
        time.sleep(3)  # Esperar que cargue la página

        # Obtener todos los <a> con href
        enlaces = driver.find_elements(By.XPATH, "//a[@href]")

        # Filtrar los href deseados
        for enlace in enlaces:
            href = enlace.get_attribute("href")
            if href and "https://www.bumeran.com.pe/empleos/" in href:
                todos_los_enlaces.append(href)

    driver.quit()

    # Eliminar duplicados y guardar en un DataFrame
    enlaces_unicos = list(set(todos_los_enlaces))
    Hipervinculos = pd.DataFrame(enlaces_unicos, columns=["Link"])

    return Hipervinculos


In [3]:
df = Tarea()
df

,Link
0,https://www.bumeran.com.pe/empleos/desarrollad...
1,https://www.bumeran.com.pe/empleos/programador...
2,https://www.bumeran.com.pe/empleos/frontend-de...
3,https://www.bumeran.com.pe/empleos/programador...
4,https://www.bumeran.com.pe/empleos/software-ar...
...,...
109,https://www.bumeran.com.pe/empleos/analista-pr...
110,https://www.bumeran.com.pe/empleos/desarrollad...
111,https://www.bumeran.com.pe/empleos/backend-dev...
112,https://www.bumeran.com.pe/empleos/analista-pr...


In [5]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException
import time

def get_text_safe(driver, xpath, timeout=3, retries=1):
    for attempt in range(retries):
        try:
            WebDriverWait(driver, timeout).until(
                EC.presence_of_element_located((By.XPATH, xpath))
            )
            element = driver.find_element(By.XPATH, xpath)
            return element.text.strip()
        except (TimeoutException, StaleElementReferenceException):
            time.sleep(1) 
    return None

def extraer_info(driver, url):
    driver.get(url)
    
    try:
        WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.ID, "header-component"))
        )
    except TimeoutException:
        print(f"[WARN] La página tardó mucho en cargar: {url}")

    return {
        "URL": url,
        "Titulo": get_text_safe(driver, '//*[@id="header-component"]/div[1]/div/div[1]/h1'),
        "Descripción": get_text_safe(driver, '//*[@id="ficha-detalle"]/div[2]/div/div[1]/p/p[1]'),
        "Distrito": get_text_safe(driver, '//*[@id="ficha-detalle"]/div[2]/div/div[1]/div[1]/div[2]/div/div/li/a/h2'),
        "Tipo": get_text_safe(driver, '//*[@id="ficha-detalle"]/div[2]/div/div[1]/div[4]/div/ul/div[1]/li[1]/a/p')
    }

def scrap_info_laboral(df_urls):
    driver = webdriver.Chrome()
    info_lista = []

    for url in df_urls['Link']:
        try:
            info = extraer_info(driver, url)
            info_lista.append(info)
        except Exception as e:
            print(f"[ERROR] {url}: {e}")

    driver.quit()
    return pd.DataFrame(info_lista)


In [7]:
Info_Laboral = scrap_info_laboral(df)

In [9]:
Info_Laboral 

,URL,Titulo,Descripción,Distrito,Tipo
0,https://www.bumeran.com.pe/empleos/desarrollad...,Desarrollador/Programador Excel (Avanzado) Fre...,Desarrollador/Programador Excel (Avanzado) Fre...,"Lima, Lima, Peru",Presencial
1,https://www.bumeran.com.pe/empleos/programador...,Programador ABAP / Preencial,En Experis Perú buscamos al mejor talento para...,"Callao, Lima, Peru",Presencial
2,https://www.bumeran.com.pe/empleos/frontend-de...,Frontend Developer React,ZUTUN. es una empresa de software boutique con...,"Pueblo Libre, Lima, Peru",Híbrido
3,https://www.bumeran.com.pe/empleos/programador...,PROGRAMADOR JUNIOR Java y Angular PRESENCIAL,¡Reinventa el futuro con Canvia!,"Lima, Lima, Peru",Presencial
4,https://www.bumeran.com.pe/empleos/software-ar...,Software Architect,"Por encargo de nuestro cliente INTERSEGURO, em...","San Isidro, Lima, Peru",Híbrido
...,...,...,...,...,...
109,https://www.bumeran.com.pe/empleos/analista-pr...,Analista Programador de Sistemas - Híbrido,"Tecsup, instituto líder en educación tecnológi...","Santa Anita, Lima, Peru",None
110,https://www.bumeran.com.pe/empleos/desarrollad...,Desarrollador Fullstack Vtex.,Creditienda es una nueva iniciativa del grupo ...,"Miraflores, Lima, Peru",Híbrido
111,https://www.bumeran.com.pe/empleos/backend-dev...,Backend Developer Java/Azure Semi Senior,Perfil:,"Lima, Lima, Peru",Híbrido
112,https://www.bumeran.com.pe/empleos/analista-pr...,Analista Programador .Net,Métrica Andina es la primera filial latinoamer...,"Chorrillos, Lima, Peru",Híbrido


In [11]:
df.to_excel("nombre_del_archivo.xlsx", index=False)

In [15]:
Info_Laboral.to_excel("Info_Laboral.xlsx", index=False)